# SASRec Eval-Seed Analysis

This notebook loads SASRec evaluation artifacts produced by `scripts/sasrec_eval.py`. It supports both eval modes: `normal` for one deterministic full-sort pass and `eval_seeds` for the RPG-style ten-seed protocol. The reported paper-style metric is `final_user_avg`: average each user across eval seeds first, then average over users.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

ROOT = Path.cwd().resolve()
while ROOT.name not in {"RPG", "RPG-uva"} and ROOT.parent != ROOT:
    ROOT = ROOT.parent

ARTIFACT_ROOTS = [
    Path("/projects/prjs2120/groups/group_16/artifacts/sasrec/eval_seeds"),
    ROOT / "artifacts" / "sasrec" / "eval_seeds",
]

DATASET_LABELS = {
    "Sports_and_Outdoors": "Sports",
    "sports_and_outdoors": "Sports",
    "Beauty": "Beauty",
    "beauty": "Beauty",
    "Toys_and_Games": "Toys",
    "toys_and_games": "Toys",
    "CDs_and_Vinyl": "CDs",
    "cds_and_vinyl": "CDs",
}
TRACK_LABELS = {"released_readme": "released README", "legacy": "legacy"}
DATASET_ORDER = ["Sports", "Beauty", "Toys", "CDs"]
EVAL_MODE_ORDER = ["normal", "eval_seeds"]
REPORT_METRIC = "ndcg@10"


In [ ]:
def _dataset_label(value: str | None) -> str:
    if value is None:
        return "Unknown"
    return DATASET_LABELS.get(value, value)


def _track_label(value: str) -> str:
    return TRACK_LABELS.get(value, value)


def _sort_runs(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame
    result = frame.copy()
    result["_dataset_rank"] = result["dataset"].map({name: idx for idx, name in enumerate(DATASET_ORDER)}).fillna(len(DATASET_ORDER))
    result["_mode_rank"] = result["eval_mode"].map({name: idx for idx, name in enumerate(EVAL_MODE_ORDER)}).fillna(len(EVAL_MODE_ORDER))
    result = result.sort_values(["_dataset_rank", "track_label", "_mode_rank", "session"])
    return result.drop(columns=["_dataset_rank", "_mode_rank"]).reset_index(drop=True)


def _parse_summary(summary_path: Path, artifact_root: Path) -> list[dict]:
    payload = json.loads(summary_path.read_text())
    relative_parts = summary_path.relative_to(artifact_root).parts
    session_root = summary_path.parent
    if len(relative_parts) >= 4:
        track = relative_parts[0]
        dataset_slug = relative_parts[1]
        session = relative_parts[2]
    else:
        track = "legacy"
        dataset_slug = payload.get("category") or payload.get("dataset") or "unknown"
        session = session_root.name

    rows = []
    for metric_row in payload.get("metric_summary", []):
        rows.append(
            {
                "artifact_root": str(artifact_root),
                "summary_path": str(summary_path),
                "session_root": str(session_root),
                "session": session,
                "track": track,
                "track_label": _track_label(track),
                "dataset_slug": dataset_slug,
                "dataset": _dataset_label(payload.get("category") or dataset_slug),
                "category": payload.get("category"),
                "model": "SASRec",
                "eval_mode": payload.get("eval_mode", "eval_seeds" if len(payload.get("eval_seeds", [])) > 1 else "normal"),
                "n_eval_seeds": len(payload.get("eval_seeds", [])),
                "eval_seeds": ",".join(str(seed) for seed in payload.get("eval_seeds", [])),
                "checkpoint_path": payload.get("checkpoint_path"),
                "metric": metric_row.get("metric"),
                "n_users": metric_row.get("n_users"),
                "final_user_avg": metric_row.get("final_user_avg"),
                "user_bootstrap_ci_low": metric_row.get("user_bootstrap_ci_low"),
                "user_bootstrap_ci_high": metric_row.get("user_bootstrap_ci_high"),
                "eval_seed_mean": metric_row.get("eval_seed_mean"),
                "eval_seed_std": metric_row.get("eval_seed_std"),
                "eval_seed_min": metric_row.get("eval_seed_min"),
                "eval_seed_max": metric_row.get("eval_seed_max"),
            }
        )
    return rows


all_rows = []
for artifact_root in ARTIFACT_ROOTS:
    if not artifact_root.exists():
        continue
    for summary_path in sorted(artifact_root.rglob("summary.json")):
        all_rows.extend(_parse_summary(summary_path, artifact_root))

runs = pd.DataFrame(all_rows)
if runs.empty:
    print("No SASRec eval summary.json files found under:")
    for root in ARTIFACT_ROOTS:
        print(f"  - {root}")
    latest_runs = pd.DataFrame()
else:
    latest_keys = (
        runs[["dataset", "track", "eval_mode", "session", "session_root"]]
        .drop_duplicates()
        .sort_values(["dataset", "track", "eval_mode", "session"])
        .groupby(["dataset", "track", "eval_mode"], as_index=False)
        .tail(1)
    )
    latest_runs = runs.merge(latest_keys[["session_root"]], on="session_root", how="inner")
    latest_runs = _sort_runs(latest_runs)

print(f"Discovered {len(runs)} metric rows across {sum(root.exists() for root in ARTIFACT_ROOTS)} artifact roots.")


## Latest NDCG@10 Runs

In [ ]:
if latest_runs.empty:
    print("No SASRec eval runs available.")
else:
    latest_metric = latest_runs[latest_runs["metric"] == REPORT_METRIC].copy()
    display(
        latest_metric[
            [
                "dataset",
                "track_label",
                "eval_mode",
                "session",
                "n_eval_seeds",
                "n_users",
                "final_user_avg",
                "eval_seed_mean",
                "eval_seed_std",
                "user_bootstrap_ci_low",
                "user_bootstrap_ci_high",
                "summary_path",
            ]
        ].rename(columns={"track_label": "track"})
    )


## Normal vs Ten-Seed Protocol

In [ ]:
if latest_runs.empty:
    print("No SASRec eval runs available.")
else:
    metric = latest_runs[latest_runs["metric"] == REPORT_METRIC].copy()
    comparison = metric.pivot_table(
        index=["dataset", "track_label"],
        columns="eval_mode",
        values="final_user_avg",
        aggfunc="first",
    ).reset_index()
    if {"normal", "eval_seeds"}.issubset(comparison.columns):
        comparison["eval_seeds_minus_normal"] = comparison["eval_seeds"] - comparison["normal"]
    display(comparison.rename(columns={"track_label": "track"}))


## Plot

In [ ]:
if latest_runs.empty:
    print("No SASRec eval runs available.")
else:
    metric = latest_runs[latest_runs["metric"] == REPORT_METRIC].copy()
    if metric.empty:
        print(f"No {REPORT_METRIC} rows available.")
    else:
        datasets = [name for name in DATASET_ORDER if name in set(metric["dataset"])]
        if not datasets:
            datasets = sorted(metric["dataset"].unique())
        fig, ax = plt.subplots(figsize=(10, 4.8), constrained_layout=True)
        labels = []
        values = []
        colors = []
        for dataset in datasets:
            for mode in EVAL_MODE_ORDER:
                row = metric[(metric["dataset"] == dataset) & (metric["eval_mode"] == mode)]
                if row.empty:
                    continue
                labels.append(f"{dataset}\n{mode}")
                values.append(float(row["final_user_avg"].iloc[0]))
                colors.append("#7A5195" if mode == "normal" else "#2A9D8F")
        ax.bar(range(len(values)), values, color=colors)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels)
        ax.set_ylabel(f"{REPORT_METRIC} final_user_avg")
        ax.set_title("SASRec Evaluation Protocol Comparison")
        ax.grid(axis="y", alpha=0.25)
        ax.set_ylim(bottom=0)
        plt.show()


## All Metrics

In [ ]:
if latest_runs.empty:
    print("No SASRec eval runs available.")
else:
    display(
        latest_runs[
            [
                "dataset",
                "track_label",
                "eval_mode",
                "metric",
                "n_eval_seeds",
                "final_user_avg",
                "eval_seed_std",
                "user_bootstrap_ci_low",
                "user_bootstrap_ci_high",
            ]
        ].rename(columns={"track_label": "track"})
    )
